# 00 — Project configuration and shared helpers

This notebook is not run on its own. Every other notebook loads it with `%run 00_setup.ipynb`,
so paths, sample periods, carrier lists and plotting style are defined once.

In [ ]:
import os, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean"
INTERIM = ROOT / "data" / "interim"      # large intermediate files, not committed
TAB = ROOT / "output" / "tables"
FIG = ROOT / "output" / "figures"
RES = ROOT / "output" / "results"
for p in (RAW, CLEAN, INTERIM, TAB, FIG, RES):
    p.mkdir(parents=True, exist_ok=True)

# Sample period: 2022Q1 through the latest DB1B quarter posted by BTS (2025Q2 at the time of writing).
FIRST_Q = (2022, 1)
LAST_Q = (2025, 2)
QUARTERS = [(y, q) for y in range(FIRST_Q[0], LAST_Q[0] + 1) for q in range(1, 5)
            if FIRST_Q <= (y, q) <= LAST_Q]

# Alaska Air Group closed its acquisition of Hawaiian Holdings on 18 September 2024.
AS_HA_PRE = [(y, q) for (y, q) in QUARTERS if (y, q) <= (2024, 2)]   # estimation / simulation window
AS_HA_TRANSITION = [(2024, 3)]                                        # closing quarter, excluded
AS_HA_POST = [(y, q) for (y, q) in QUARTERS if (y, q) >= (2024, 4)]

# Carriers kept as separate brands; all other ticketing carriers are pooled as "OTH" for fixed effects.
MAJOR_CARRIERS = ["AA", "DL", "UA", "WN", "AS", "B6", "NK", "F9", "G4", "HA", "SY", "MX", "XP"]
CARRIER_NAMES = {"AA": "American", "DL": "Delta", "UA": "United", "WN": "Southwest", "AS": "Alaska",
                 "B6": "JetBlue", "NK": "Spirit", "F9": "Frontier", "G4": "Allegiant", "HA": "Hawaiian",
                 "SY": "Sun Country", "MX": "Breeze", "XP": "Avelo", "OTH": "Other"}

def qlabel(y, q):
    return f"{y}Q{q}"

def qindex(y, q):
    """Consecutive integer index of a quarter (2022Q1 = 0)."""
    return (np.asarray(y) - FIRST_Q[0]) * 4 + (np.asarray(q) - 1)

## Plot and table style

In [ ]:
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "serif", "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.3, "legend.frameon": False,
})
PALETTE = ["#1f4e79", "#c0504d", "#4f8f3a", "#8064a2", "#d18f00", "#2c8d9b", "#7f7f7f"]
mpl.rcParams["axes.prop_cycle"] = mpl.cycler(color=PALETTE)

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}")
    print("saved", FIG / f"{name}.pdf")

def save_table(df, name, caption=None, label=None, float_format="%.3f", index=True, note=None, **kw):
    """Write a booktabs LaTeX table (and a CSV copy) to output/tables."""
    df.to_csv(TAB / f"{name}.csv", index=index)
    body = df.to_latex(index=index, float_format=float_format, escape=True, **kw)
    body = body.replace("\\toprule", "\\toprule").replace("\\bottomrule", "\\bottomrule")
    if note:
        body = body.replace("\\end{tabular}", "\\end{tabular}\n\\par\\vspace{2pt}{\\footnotesize " + note + "}")
    (TAB / f"{name}.tex").write_text(body)
    print("saved", TAB / f"{name}.tex")

# SMOKE=1 runs every notebook end to end on small samples; its result files get a "_smoke" suffix.
SMOKE = bool(int(os.environ.get("SMOKE", "0")))

def save_json(obj, name):
    (RES / f"{name}{'_smoke' if SMOKE else ''}.json").write_text(json.dumps(obj, indent=2, default=float))

def load_json(name):
    return json.loads((RES / f"{name}{'_smoke' if SMOKE else ''}.json").read_text())